In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dataclasses import replace

from src.config import SimulationConfig
from src.simulator import run_simulation

In [2]:
base_config = SimulationConfig(
    nx=128,
    ny=128,
    eta=1.5,
    correlation_length=5,
    weibull_shape=5.0,
    mean_breakdown_field=4.0,
    random_seed=42,
    max_growth_steps=5000
)

disorder_seeds = range(50)

In [3]:
thresholds = [3.0, 3.2, 3.4, 3.6, 3.8, 4.0]

threshold_records = []

for threshold in thresholds:
    for disorder_seed in disorder_seeds:
        config = replace(
            base_config,
            mean_breakdown_field=threshold,
            disorder_seed=disorder_seed,
        )

        field, history, breakdown = run_simulation(config)

        if breakdown:
            status = "breakdown"
        elif len(history) == 0:
            status = "no_start"
        elif len(history) >= config.max_growth_steps:
            status = "max_steps"
        else:
            status = "arrested"

        threshold_records.append(
            {
                "threshold": threshold,
                "disorder_seed": disorder_seed,
                "status": status,
                "breakdown": breakdown,
            }
        )

In [4]:
threshold_df = pd.DataFrame(threshold_records)

summary = (
    threshold_df
    .groupby("threshold")
    .agg(
        p_breakdown=("breakdown", "mean"),
        n=("breakdown", "size"),
    )
)

summary

,p_breakdown,n
threshold,,
3.0,0.52,50
3.2,0.36,50
3.4,0.24,50
3.6,0.20,50
3.8,0.14,50
4.0,0.10,50


### Выбор рабочего порога

По результатам pilot-sweep выбираем

$$
E_{bd}^{(0)} = 3.0.
$$

При этом значении вероятность полного пробоя составляет примерно

$$
P_{breakdown} \approx 0.52.
$$

То есть наблюдаются как полные пробои, так и остановки канала. Фиксируем все параметры модели и `random_seed`, а меняем только `disorder_seed`.

Для получения более устойчивых статистических оценок используем 200 независимых реализаций disorder.

In [5]:
base_config = replace(
    base_config,
    mean_breakdown_field=3.0,
    random_seed=42,
)

disorder_seeds = range(200)

records = []

In [6]:
for disorder_seed in disorder_seeds:
    config = replace(
        base_config,
        disorder_seed=disorder_seed,
    )

    field, history, breakdown = run_simulation(config)

    if breakdown:
        status = "breakdown"
    elif len(history) == 0:
        status = "no_start"
    elif len(history) >= config.max_growth_steps:
        status = "max_steps"
    else:
        status = "arrested"

    if np.any(field.channel):
        max_y = int(np.max(np.where(field.channel)[0]))
    else:
        max_y = 0

    height_fraction = max_y / (config.ny - 1)

    records.append(
        {
            "disorder_seed": disorder_seed,
            "status": status,
            "breakdown": breakdown,
            "steps": len(history),
            "max_y": max_y,
            "height_fraction": height_fraction,
        }
    )

In [7]:
results_df = pd.DataFrame(records)

results_df["status"].value_counts()

status
breakdown    106
no_start      93
arrested       1
Name: count, dtype: int64

In [8]:
from scipy.stats import norm

n = len(results_df)
k = results_df["breakdown"].sum()

p = k / n

z = norm.ppf(0.975)

denominator = 1 + z**2 / n

center = (
    p + z**2 / (2 * n)
) / denominator

half_width = (
    z
    * np.sqrt(
        p * (1 - p) / n
        + z**2 / (4 * n**2)
    )
    / denominator
)

ci_low = center - half_width
ci_high = center + half_width

print("P(breakdown):", p)
print("95% CI:", ci_low, ci_high)

P(breakdown): 0.53
95% CI: 0.4609168291500183 0.5979524512673458


In [9]:
results_df["started"] = results_df["status"] != "no_start"

p_start = results_df["started"].mean()

started_df = results_df[results_df["started"]]
p_breakdown_given_start = started_df["breakdown"].mean()

print("P(start):", p_start)
print("P(breakdown | start):", p_breakdown_given_start)

P(start): 0.535
P(breakdown | start): 0.9906542056074766


In [10]:
from src.field import ElectricField
from src.disorder import generate_breakdown_strength
from src.growth import get_front

initial_field = ElectricField(base_config)
initial_field.solve_laplace_sor_numba()

_, _, E = initial_field.electric_field()

front = get_front(
    initial_field.needle | initial_field.channel
)

E_front = E[front]

In [11]:
disorder_features = []

for disorder_seed in disorder_seeds:
    strength = generate_breakdown_strength(
        ny=base_config.ny,
        nx=base_config.nx,
        correlation_length=base_config.correlation_length,
        weibull_shape=base_config.weibull_shape,
        seed=disorder_seed,
    )

    S_front = strength[front]

    max_drive = np.max(E_front / S_front)

    disorder_features.append(
        {
            "disorder_seed": disorder_seed,
            "front_strength_mean": S_front.mean(),
            "front_strength_min": S_front.min(),
            "max_drive": max_drive,
        }
    )

In [12]:
features_df = pd.DataFrame(disorder_features)

results_df = results_df.merge(
    features_df,
    on="disorder_seed"
)

results_df.head()

,disorder_seed,status,breakdown,steps,max_y,height_fraction,started,front_strength_mean,front_strength_min,max_drive
0,0,breakdown,True,345,127,1.000000,True,0.861997,0.568417,5.113771
1,1,no_start,False,0,0,0.000000,False,0.805891,0.428039,2.520573
2,2,breakdown,True,338,127,1.000000,True,0.762357,0.445065,3.603933
3,3,no_start,False,0,0,0.000000,False,0.881376,0.455690,2.578746
4,4,arrested,False,3,16,0.125984,True,0.785207,0.667698,3.097158


In [13]:
results_df["predicted_start"] = (
    results_df["max_drive"]
    > base_config.mean_breakdown_field
)

pd.crosstab(
    results_df["started"],
    results_df["predicted_start"]
)

predicted_start,False,True
started,,
False,93,0
True,0,107


In [14]:
results_df.groupby("started")[
    [
        "front_strength_mean",
        "front_strength_min",
        "max_drive",
    ]
].agg(["mean", "std", "median"])

front_strength_mean                     front_strength_min            \
                       mean       std    median               mean       std   
started                                                                        
False              1.058837  0.167188  1.072345           0.797201  0.252690   
True               0.942232  0.146607  0.955822           0.662076  0.174883   

                  max_drive                      
           median      mean       std    median  
started                                          
False    0.845841  2.558494  0.246405  2.565123  
True     0.684986  3.758835  0.912879  3.437530

In [15]:
from scipy.stats import permutation_test

started = results_df.loc[
    results_df["started"],
    "front_strength_mean"
].values

not_started = results_df.loc[
    ~results_df["started"],
    "front_strength_mean"
].values

def mean_diff(x, y):
    return np.mean(x) - np.mean(y)

test = permutation_test(
    (started, not_started),
    mean_diff,
    permutation_type="independent",
    n_resamples=10_000,
    alternative="two-sided",
    random_state=42,
)

print("difference:", mean_diff(started, not_started))
print("p-value:", test.pvalue)

difference: -0.11660494756305417
p-value: 0.00019998000199980003


## Вывод

В этом ноутбуке было отдельно исследовано влияние случайной реализации неоднородности материала через параметр `disorder_seed`.

При фиксированных физических параметрах и фиксированном `random_seed` было рассмотрено 200 различных реализаций disorder. Для рабочего порога

$$
E_{bd}^{(0)} = 3.0
$$

получено

$$
P_{breakdown} \approx 0.53.
$$

Из 200 реализаций 106 завершились полным пробоем, 93 не дали старта канала и только 1 реализация привела к остановке уже начавшегося роста.

Также было показано, что условие старта определяется локальным соотношением поля и прочности материала на начальном фронте. Предсказание старта по критерию

$$
\max_i \frac{E_i}{S_i} > E_{bd}^{(0)}
$$

полностью совпало с результатами симуляции.

Permutation test показал, что средняя прочность материала на фронте в случаях старта статистически значимо ниже:

$$
\Delta \approx -0.117,
\qquad
p \approx 2 \cdot 10^{-4}.
$$

Таким образом, случайная пространственная структура материала существенно влияет на возможность возникновения пробоя.

### Дальнейший план

Следующий этап — перейти от анализа отдельных реализаций disorder к исследованию физических параметров модели неоднородности:

- `weibull_shape = k`;
- `correlation_length`;
- затем параметра роста `eta`;
- и базового порога пробоя.

Для каждого параметра будет использоваться ансамбль различных `disorder_seed`, после чего будут оцениваться вероятность старта, вероятность полного пробоя и характеристики выросшего канала.

После определения информативных диапазонов параметров эти эксперименты будут использованы для генерации большого датасета для последующего ML-анализа.